<a href="https://colab.research.google.com/github/chivian/Natural_Language_Processing/blob/master/nlp_lab3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Note: We will start by importing TensorFlow and preparing our text data. Neural networks require text to be converted into sequences of numbers (tokens) and padded so every review is the exact same length.

In [ ]:
# Cell 1: Sequence Preparation for Neural Networks
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from datasets import load_dataset

# Quick data load and split (grabbing 2000 random rows for speed)
dataset = load_dataset("rotten_tomatoes")
df = pd.DataFrame(dataset['train']).sample(n=2000, random_state=42).copy()

# Hyperparameters for tokenisation
vocab_size = 5000
max_length = 50
trunc_type = 'post'
padding_type = 'post'

# Initialise and fit the Tokeniser
tokeniser = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokeniser.fit_on_texts(df['text'])

# Convert text to sequences of integers
sequences = tokeniser.texts_to_sequences(df['text'])

# Pad the sequences so they are all 'max_length' long
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

print("--- Sequence Preparation Complete ---")
print(f"Original text: {df['text'].iloc[0]}")
print(f"Padded sequence: {padded_sequences[0]}")

--- Sequence Preparation Complete ---
Original text: it would take a complete moron to foul up a screen adaptation of oscar wilde's classic satire .
Padded sequence: [  10   92  148    3  374 3259    6 1345   42    3  149  503    5  504
 1346  397  675    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0]


Note: Now we build the RNN. We use an Embedding layer to learn the meaning of words, an LSTM layer to learn the sequential order, and a final Dense layer to output a probability (0 to 1).

In [ ]:
# Cell 2: Building and Training the LSTM Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Define the LSTM architecture
model = Sequential([
    Embedding(vocab_size, 64, input_length=max_length),
    LSTM(64),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid') # Sigmoid outputs a value between 0 (Negative) and 1 (Positive)
])

# Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

print(model.summary())

# Train the model (using a small number of epochs for lab speed)
print("\nTraining the LSTM Model...")
labels = np.array(df['label'])
history = model.fit(padded_sequences, labels, epochs=3, validation_split=0.2, verbose=1)
print("Training Complete!")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None

Training the LSTM Model...
Epoch 1/3
50/50 ━━━━━━━━━━━━━━━━━━━━ 9s 74ms/step - accuracy: 0.5034 - loss: 0.6943 - val_accuracy: 0.5400 - val_loss: 0.6925
Epoch 2/3
50/50 ━━━━━━━━━━━━━━━━━━━━ 4s 63ms/step - accuracy: 0.4732 - loss: 0.6936 - val_accuracy: 0.5400 - val_loss: 0.6914
Epoch 3/3
50/50 ━━━━━━━━━━━━━━━━━━━━ 4s 43ms/step - accuracy: 0.5116 - loss: 0.6928 - val_accuracy: 0.4600 - val_loss: 0.6934
Training Complete!


Note: Next, we leap to state-of-the-art NLP. Instead of training from scratch, we will download a pre-trained Transformer model from Hugging Face. This requires almost zero setup and yields incredibly accurate results.

In [ ]:
# Cell 3: Initialising a Pre-trained Transformer
from transformers import pipeline

# Initialise a sentiment analysis pipeline using a default Transformer model (DistilBERT)
print("Downloading and initialising the Transformer model...")
transformer_classifier = pipeline("sentiment-analysis")

print("Transformer initialised successfully!")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Transformer initialised successfully!


Note: Let us test both models on a tricky sentence that requires deep contextual understanding.

In [ ]:
# Cell 4: Comparative Analysis
test_sentence = "I really wanted to hate this movie, and the acting was mostly terrible, but somehow I ended up loving the story."

# 1. Get prediction from our custom LSTM
# We must tokenise and pad the new sentence first
test_seq = tokeniser.texts_to_sequences([test_sentence])
test_padded = pad_sequences(test_seq, maxlen=max_length, padding=padding_type, truncating=trunc_type)
lstm_pred = model.predict(test_padded)[0][0]
lstm_sentiment = "Positive" if lstm_pred > 0.5 else "Negative"

# 2. Get prediction from the Hugging Face Transformer
transformer_result = transformer_classifier(test_sentence)[0]

print("\n--- Comparative Results ---")
print(f"Test Sentence: '{test_sentence}'\n")

print("LSTM Prediction:")
print(f"Sentiment: {lstm_sentiment} (Confidence score: {lstm_pred:.4f})")

print("\nTransformer Prediction:")
print(f"Sentiment: {transformer_result['label']} (Confidence score: {transformer_result['score']:.4f})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 207ms/step

--- Comparative Results ---
Test Sentence: 'I really wanted to hate this movie, and the acting was mostly terrible, but somehow I ended up loving the story.'

LSTM Prediction:
Sentiment: Negative (Confidence score: 0.4984)

Transformer Prediction:
Sentiment: POSITIVE (Confidence score: 0.9977)
